## Question 2 - *What is the percentage of computational power lost due to maintenance (a machine went offline and reconnected later)?*
#### The computational power is proportional to both the CPU capacity and the unavailability period of machines.

According to the **Google Document**, there are three type of events in *machine_events*:
- *ADD* a machine to the cluster (0)
- *REMOVE* a machine from the cluster (1)
- *UPDATE* the capacity of a machine (2)

By definition, a machine goes offline when its *event_type* is 1 in *machine_events*. <br>
To answer this question, we need:
- Timestamp (field 0)
- Machine_ID (field 1)
- Event_type (field 2)
- CPU capacity (field 4)

Since computational power is proportional to CPU capacity and time, we calculate:
1. **Total alive time**: Time from first event to last event for each machine
2. **Total offline time**: Cumulative time when machines are offline (between event_type 1→0 transitions)
3. Weight both by CPU capacity (assuming it remains constant per machine)

The percentage of computational power lost is:
```
(total_offline_time × CPU_capacity) / (total_existence_time × CPU_capacity) × 100
```

In [ ]:
import sys
from pyspark import SparkContext

# Initialising Spark with 1 worker thread
sc = SparkContext("local[1]")

machine_events = sc.textFile("./data/machine_events/part-00000-of-00001.csv.gz")

# Calculating total time alive 
machines_events_1_024 = (
    machine_events
    .map(lambda line: line.split(","))          # converting strings into fields
    .filter(lambda x: x[4] != '')
    .map(lambda x: (x[1],(int(x[0]),int(x[2]),float(x[4]))))     # key,value : ( machine_ID, (timestamp, event_type, cpu_capacityacity) )
)

total_alive_CPUs = (
    machines_events_1_024
    .map(lambda x: (x[0], (x[1][0], x[1][0], x[1][2]))) # (machine_ID , (timestamp, timestamp, CPU_capacity) )
    .reduceByKey(lambda a, b: ( min(a[0], b[0]), max(a[1], b[1]), a[2] )) # keeping CPU
    .map(lambda x: (x[1][1] - x[1][0]) * x[1][2])                         # time_alive * CPU_capacity
    .sum()
)

# Calculating total time offline

# Fuction used to calculate total offline time
def process_machine(machine_id, events):
        sorted_events = sorted(list(events), key=lambda x: x[0])  # sorts the events (timestamp, event_type, CPU_capacity) by timestamp
        
        total_offline_time = 0
        offline_start = None
        prev_event_type = None
        
        for timestamp, event_type, cpu_capacity in sorted_events:    # looping and unpacking
            # Goes from online (1) to offline (0)
            if prev_event_type == 1 and event_type == 0:
                offline_start = timestamp
            # Goes from offline (0) to online (1)
            elif prev_event_type == 0 and event_type == 1 and offline_start is not None:
                total_offline_time = total_offline_time + (timestamp - offline_start)
                offline_start = None
            
            prev_event_type = event_type
        
        return (machine_id, total_offline_time, cpu_capacity)
    
total_offline_CPUs = (
    machines_events_1_024           # key,value : ( machine_ID, (timestamp, event_type, CPU_capacity) )
    .filter(lambda x: x[1][1]!=2)   # exclude the UPDATE events
    .groupByKey()
    .map(lambda x: (process_machine(x[0], x[1]))) # (machine_id, total_offline_time, cpu_capacity)
    .map(lambda x: (x[1]*x[2]))     # time_offline * CPU_capacity
    .sum()
)

# Printing the result
result = (total_offline_CPUs / total_alive_CPUs) * 100
print("\n" + "="*80)
print(f"Percentage of lost computational power: {result}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/14 23:01:19 WARN Utils: Your hostname, im2ag-mandelbrot, resolves to a loopback address: 127.0.1.1; using 152.77.81.20 instead (on interface ens18)
26/01/14 23:01:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/14 23:01:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/14 23:01:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.



Percentage of lost computational power: 18.462309039750348
